# 🎬 Исследование и обучение 3D Conditional VAE для генерации видео ИК-термографии

**Цель:** Обучение трехмерной вариационной автоэнкодерной сети (3D-CVAE) на пространственно-временных кубах дефектных зон (`86x86x3000`, ресемплированных до `300` кадров).

**Входные данные:** `.npy` файлы из `data/processed_3d/train/`.
**Выходные данные:** Веса обученной модели в `checkpoints/cvae_3d_weights.pth`.
**Оборудование:** Yandex DataSphere GPU (`cuda`).

In [1]:
import sys
import os
from pathlib import Path
import torch
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# Автоматическое определение корня проекта
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Импортируем наш бэкенд из src
from src.dataloaders.video_dataset import get_video_dataloader
from src.models.cvae_3d import VideoConditionalVAE

print(f"📁 Корень проекта: {project_root}")

📁 Корень проекта: C:\programming\studcamp\project\honeycomb-water-detection


In [ ]:
# =====================================================================
# ⚙️ ГИПЕРПАРАМЕТРЫ ЭКСПЕРИМЕНТА
# =====================================================================
DATA_3D_DIR = project_root / "data" / "processed_3d" / "train"
CHECKPOINT_DIR = project_root / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = CHECKPOINT_DIR / "cvae_3d_weights.pth"

BATCH_SIZE = 4            # Оптимально для GPU (VRAM ~8-16 GB)
EPOCHS = 150              # Количество эпох обучения
LEARNING_RATE = 1e-4      # Скорость обучения Adam
TIME_DOWNSAMPLE = 10      # Коэффициент прореживания (3000 -> 300 кадров)
TARGET_FRAMES = 300       # Финальное число кадров в 3D тензоре
LATENT_DIM = 128          # Размерность скрытого пространства z
BETA_KL = 0.05            # Вес KL-дивергенции в функции потерь

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Вычисления будут выполняться на: {DEVICE}")

In [ ]:
def loss_function_3d_cvae(recon_x, x, mu, logvar, beta=0.05):
    """
    Расчет ошибки 3D-CVAE:
    1. Reconstruction Loss (MSE) - точность восстановления температурного куба.
    2. KL Divergence - регулярность скрытого пространства z.
    """
    # Среднеквадратичная ошибка по всем пикселям и кадрам батча
    recon_loss = F.mse_loss(recon_x, x, reduction='sum') / x.size(0)
    
    # KL-дивергенция: N(mu, std) vs N(0, I)
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    
    total_loss = recon_loss + beta * kld_loss
    return total_loss, recon_loss, kld_loss

In [ ]:
print("📦 Загрузка 3D датасета...")
train_loader = get_video_dataloader(
    data_dir=DATA_3D_DIR,
    batch_size=BATCH_SIZE,
    time_downsample=TIME_DOWNSAMPLE,
    shuffle=True
)

print(f"✅ Даталоадер готов. Размеров батчей: {len(train_loader)}")

# Инициализируем 3D CVAE модель
model = VideoConditionalVAE(
    target_frames=TARGET_FRAMES,
    cond_dim=1,
    latent_dim=LATENT_DIM
).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

print(f"🧠 Модель 3D-CVAE успешно перемещена на {DEVICE}")

In [ ]:
history = {'total_loss': [], 'recon_loss': [], 'kl_loss': []}

print("🔥 Запуск обучения 3D-генератора видео...")

for epoch in range(EPOCHS):
    model.train()
    running_total, running_recon, running_kl = 0.0, 0.0, 0.0
    
    for batch_videos, batch_targets in train_loader:
        # Перенос батча на GPU: (Batch, 1, 300, 86, 86)
        batch_videos = batch_videos.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Прямой проход (Forward pass)
        recon_videos, mu, logvar = model(batch_videos, batch_targets)
        
        # Считаем лосс
        loss, recon_loss, kl_loss = loss_function_3d_cvae(
            recon_videos, batch_videos, mu, logvar, beta=BETA_KL
        )
        
        # Обратный проход (Backprop)
        loss.backward()
        optimizer.step()
        
        running_total += loss.item()
        running_recon += recon_loss.item()
        running_kl += kl_loss.item()
        
    # Усреднение показателей за эпоху
    epoch_total = running_total / len(train_loader)
    epoch_recon = running_recon / len(train_loader)
    epoch_kl = running_kl / len(train_loader)
    
    history['total_loss'].append(epoch_total)
    history['recon_loss'].append(epoch_recon)
    history['kl_loss'].append(epoch_kl)
    
    # Логирование
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Эпоха [{epoch+1:03d}/{EPOCHS}] | Total Loss: {epoch_total:.2f} | Recon (MSE): {epoch_recon:.2f} | KL: {epoch_kl:.2f}")

# Сохранение обученных весов
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"\n💾 Веса модели успешно сохранены в: {MODEL_SAVE_PATH}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График Total & Reconstruction Loss
ax1.plot(history['total_loss'], label='Total Loss', color='purple', linewidth=2)
ax1.plot(history['recon_loss'], label='Reconstruction (MSE)', color='blue', linestyle='--')
ax1.set_title('Сходимость ошибки 3D-CVAE')
ax1.set_xlabel('Эпохи')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# График KL Divergence
ax2.plot(history['kl_loss'], label='KL Divergence', color='orange', linewidth=2)
ax2.set_title('Регуляризация латентного пространства (KL)')
ax2.set_xlabel('Эпохи')
ax2.set_ylabel('KL Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()